In [51]:
import pandas as pd

In [52]:
CATALOGUE_PATH = "../data/processed/metadata/segregatedMergedCatalogue.txt"

BAD_BUBBLES_PATH = "../data/processed/metadata/badBubbles.txt"

OUTPUT_PATH = "../data/processed/metadata/cleanedSegregatedMergedCatalogue.txt"


In [53]:
df = pd.read_csv(CATALOGUE_PATH)

print("Total bubbles before cleaning:", len(df))
df.head()

Total bubbles before cleaning: 113


,GLOBAL_ID,ID,RA_DMS,DEC_DMS,SEMI_MAJ_PC,SEMI_MIN_PC,AVG_RAD_PC,PA_DEG,ARM,DIST_ARM_PC,GAL_RAD_KPC,CATALOGUE
0,10,10,24d08m59.3s,15d48m6.1s,209,171,196,0,3,112,5.04,A
1,12,12,24d09m0.4s,15d47m52.2s,163,93,139,20,3,80,4.62,A
2,26,26,24d09m4.3s,15d48m11.1s,204,142,183,70,3,83,5.02,A
3,44,44,24d09m12.5s,15d48m12.1s,144,91,127,-60,3,23,4.78,A
4,74,74,24d09m17.3s,15d47m3.6s,129,87,115,20,3,7,3.15,A


In [54]:
bad_df = pd.read_csv(BAD_BUBBLES_PATH)

print("Bad bubbles listed:", len(bad_df))
bad_df.head()

Bad bubbles listed: 24


,GLOBAL_ID
0,26
1,110
2,255
3,282
4,2345


In [55]:
# df housekeeping

df.columns = df.columns.str.strip()
df["GLOBAL_ID"] = df["GLOBAL_ID"].astype(int)

bad_df.columns = bad_df.columns.str.strip()
bad_df["GLOBAL_ID"] = bad_df["GLOBAL_ID"].astype(int)

bad_df_unique = bad_df.drop_duplicates(subset=["GLOBAL_ID"])

print("Unique bad bubbles:", len(bad_df_unique))

# checking bad bubbles matching df_all
matched_bad = bad_df_unique.merge(
    df["GLOBAL_ID"],
    on=["GLOBAL_ID"],
    how="inner"
)

unmatched_bad = bad_df_unique.merge(
    df["GLOBAL_ID"],
    on=["GLOBAL_ID"],
    how="left",
    indicator=True
)

unmatched_bad = unmatched_bad[unmatched_bad["_merge"] == "left_only"]

print("Matching bad bubbles:", len(matched_bad))
print("Unmatched bad bubbles:", len(unmatched_bad))

Unique bad bubbles: 22
Matching bad bubbles: 22
Unmatched bad bubbles: 0


In [56]:
df_clean = df.merge(
    bad_df_unique["GLOBAL_ID"],
    on=["GLOBAL_ID"],
    how="left",
    indicator=True
)

df_clean = df_clean[df_clean["_merge"] == "left_only"].drop(columns=["_merge"])

print("Total bubbles after cleaning:", len(df_clean))
print("Removed bubbles:", len(df) - len(df_clean))

Total bubbles after cleaning: 91
Removed bubbles: 22


In [58]:
df_clean.to_csv(OUTPUT_PATH, index=False)

print("Saved full collective catalogue to", OUTPUT_PATH)

Saved full collective catalogue to ../data/processed/metadata/cleanedSegregatedMergedCatalogue.txt
